In [1]:
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
ROBOFLOW_API_KEY = user_secrets.get_secret("ROBOFLOW_API_KEY")

print("Clé Roboflow récupérée :", bool(ROBOFLOW_API_KEY))

Clé Roboflow récupérée : True


In [2]:
!pip install -q roboflow

from roboflow import Roboflow

rf = Roboflow(api_key=ROBOFLOW_API_KEY)

project = (
    rf.workspace("ayas-workspace-5bvpw")
    .project("container-shipping-number2-fdaoa")
)

version = project.version(1)
dataset = version.download("yolov11")

print("Dataset téléchargé dans :", dataset.location)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 276.9/276.9 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 38.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 53.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 92.1 MB/s eta 0:00:00
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Container-Shipping-Number2-1 in yolov11:: 100%|██████████| 3866/3866 [00:00<00:00, 8382.60it/s]

Dataset téléchargé dans : /kaggle/working/Container-Shipping-Number2-1


In [3]:
from pathlib import Path
import yaml

DATASET_DIR = Path("/kaggle/working/Container-Shipping-Number2-1")
DATA_YAML = DATASET_DIR / "data.yaml"

with open(DATA_YAML, "r", encoding="utf-8") as file:
    config = yaml.safe_load(file)

print("Configuration actuelle :")
print(config)

print("\nNombre de classes :", config.get("nc"))
print("Classes :", config.get("names"))

Configuration actuelle :
{'train': '../train/images', 'val': '../valid/images', 'test': '../test/images', 'nc': 2, 'names': ['container-number', 'iso-type'], 'roboflow': {'workspace': 'ayas-workspace-5bvpw', 'project': 'container-shipping-number2-fdaoa', 'version': 1, 'license': 'CC BY 4.0', 'url': 'https://app.roboflow.com/ayas-workspace-5bvpw/container-shipping-number2-fdaoa/1'}}

Nombre de classes : 2
Classes : ['container-number', 'iso-type']


In [4]:
from pathlib import Path
import random
import cv2
import matplotlib.pyplot as plt

DATASET_DIR = Path("/kaggle/working/Container-Shipping-Number2-1")
TRAIN_IMAGES_DIR = DATASET_DIR / "train" / "images"
TRAIN_LABELS_DIR = DATASET_DIR / "train" / "labels"

CLASS_NAMES = {
    0: "container-number",
    1: "iso-type",
}

image_paths = list(TRAIN_IMAGES_DIR.glob("*"))

random.seed(42)
selected_images = random.sample(image_paths, min(9, len(image_paths)))

plt.figure(figsize=(18, 15))

for index, image_path in enumerate(selected_images, start=1):
    image = cv2.imread(str(image_path))
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    height, width = image.shape[:2]
    label_path = TRAIN_LABELS_DIR / f"{image_path.stem}.txt"

    if label_path.exists():
        with open(label_path, "r", encoding="utf-8") as file:
            for line in file:
                parts = line.strip().split()

                if len(parts) != 5:
                    continue

                class_id = int(float(parts[0]))
                x_center, y_center, box_width, box_height = map(float, parts[1:])

                x1 = int((x_center - box_width / 2) * width)
                y1 = int((y_center - box_height / 2) * height)
                x2 = int((x_center + box_width / 2) * width)
                y2 = int((y_center + box_height / 2) * height)

                label = CLASS_NAMES.get(class_id, f"class-{class_id}")

                cv2.rectangle(image, (x1, y1), (x2, y2), (255, 0, 0), 2)
                cv2.putText(
                    image,
                    label,
                    (x1, max(y1 - 8, 15)),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.6,
                    (255, 0, 0),
                    2,
                )

    plt.subplot(3, 3, index)
    plt.imshow(image)
    plt.title(image_path.name)
    plt.axis("off")

plt.tight_layout()
plt.show()

<Figure size 1800x1500 with 9 Axes>

In [5]:
from pathlib import Path
from collections import Counter

DATASET_DIR = Path("/kaggle/working/Container-Shipping-Number2-1")

CLASS_NAMES = {
    0: "container-number",
    1: "iso-type",
}

global_counts = Counter()

for split in ["train", "valid", "test"]:
    images_dir = DATASET_DIR / split / "images"
    labels_dir = DATASET_DIR / split / "labels"

    image_files = [
        path
        for path in images_dir.iterdir()
        if path.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"}
    ]

    label_files = list(labels_dir.glob("*.txt"))

    class_counts = Counter()
    images_with_container_number = 0
    images_with_iso_type = 0
    empty_label_files = 0
    missing_label_files = 0
    invalid_lines = 0

    for image_path in image_files:
        label_path = labels_dir / f"{image_path.stem}.txt"

        if not label_path.exists():
            missing_label_files += 1
            continue

        lines = [
            line.strip()
            for line in label_path.read_text(encoding="utf-8").splitlines()
            if line.strip()
        ]

        if not lines:
            empty_label_files += 1
            continue

        classes_in_image = set()

        for line in lines:
            parts = line.split()

            if len(parts) != 5:
                invalid_lines += 1
                continue

            try:
                class_id = int(float(parts[0]))
            except ValueError:
                invalid_lines += 1
                continue

            class_counts[class_id] += 1
            global_counts[class_id] += 1
            classes_in_image.add(class_id)

        if 0 in classes_in_image:
            images_with_container_number += 1

        if 1 in classes_in_image:
            images_with_iso_type += 1

    print(f"\n===== {split.upper()} =====")
    print("Images :", len(image_files))
    print("Fichiers labels :", len(label_files))

    for class_id, class_name in CLASS_NAMES.items():
        print(f"Boîtes {class_name} :", class_counts[class_id])

    print("Images avec container-number :", images_with_container_number)
    print("Images avec iso-type :", images_with_iso_type)
    print("Labels absents :", missing_label_files)
    print("Labels vides :", empty_label_files)
    print("Lignes invalides :", invalid_lines)

print("\n===== TOTAL DES BOÎTES =====")
for class_id, class_name in CLASS_NAMES.items():
    print(f"{class_name} :", global_counts[class_id])

container_count = global_counts[0]
iso_type_count = global_counts[1]

if container_count > 0:
    print(
        "\nRatio iso-type / container-number :",
        round(iso_type_count / container_count, 3),
    )


===== TRAIN =====
Images : 1779
Fichiers labels : 1779
Boîtes container-number : 1821
Boîtes iso-type : 1611
Images avec container-number : 1779
Images avec iso-type : 1575
Labels absents : 0
Labels vides : 0
Lignes invalides : 0

===== VALID =====
Images : 115
Fichiers labels : 115
Boîtes container-number : 115
Boîtes iso-type : 102
Images avec container-number : 115
Images avec iso-type : 102
Labels absents : 0
Labels vides : 0
Lignes invalides : 0

===== TEST =====
Images : 37
Fichiers labels : 37
Boîtes container-number : 37
Boîtes iso-type : 36
Images avec container-number : 37
Images avec iso-type : 36
Labels absents : 0
Labels vides : 0
Lignes invalides : 0

===== TOTAL DES BOÎTES =====
container-number : 1973
iso-type : 1749

Ratio iso-type / container-number : 0.886


In [6]:
!pip install -q -U ultralytics

import ultralytics
from ultralytics import YOLO

print("Version Ultralytics :", ultralytics.__version__)
ultralytics.checks()

Ultralytics 8.4.104 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
Setup complete ✅ (4 CPUs, 31.3 GB RAM, 7035.8/8062.4 GB disk)


In [7]:
from pathlib import Path
from ultralytics import YOLO

DATA_YAML = Path(
    "/kaggle/working/Container-Shipping-Number2-1/data.yaml"
)

TRAINING_PROJECT = Path("/kaggle/working/runs")
TRAINING_NAME = "container_code_type_yolo11n_v2"

# Modèle préentraîné officiel YOLO11 Nano
model_v2 = YOLO("yolo11n.pt")

training_results_v2 = model_v2.train(
    data=str(DATA_YAML),
    epochs=70,
    patience=15,
    imgsz=640,
    batch=16,
    device=0,
    workers=4,
    project=str(TRAINING_PROJECT),
    name=TRAINING_NAME,
    exist_ok=False,
    pretrained=True,
    seed=42,
    deterministic=True,
    plots=True,
    verbose=True,
)

print("Entraînement V2 terminé.")
print("Dossier du run :", training_results_v2.save_dir)

Ultralytics 8.4.104 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/Container-Shipping-Number2-1/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=70, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=container_code_type_yolo11n

In [8]:
from pathlib import Path
from ultralytics import YOLO

BEST_MODEL_PATH = Path(
    "/kaggle/working/runs/container_code_type_yolo11n_v2/weights/best.pt"
)

DATA_YAML = Path(
    "/kaggle/working/Container-Shipping-Number2-1/data.yaml"
)

TEST_RUN_DIR = Path("/kaggle/working/runs")
TEST_RUN_NAME = "container_code_type_yolo11n_v2_test"

model_v2_best = YOLO(str(BEST_MODEL_PATH))

test_metrics_v2 = model_v2_best.val(
    data=str(DATA_YAML),
    split="test",
    imgsz=640,
    batch=16,
    device=0,
    workers=4,
    project=str(TEST_RUN_DIR),
    name=TEST_RUN_NAME,
    exist_ok=False,
    plots=True,
    verbose=True,
)

print("\nÉvaluation TEST terminée")
print("Précision globale :", test_metrics_v2.box.mp)
print("Recall global :", test_metrics_v2.box.mr)
print("mAP50 global :", test_metrics_v2.box.map50)
print("mAP50-95 global :", test_metrics_v2.box.map)
print("Dossier :", test_metrics_v2.save_dir)

Ultralytics 8.4.104 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
YOLO11n summary (fused): 101 layers, 2,582,542 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 918.7±299.3 MB/s, size: 37.0 KB)
val: Scanning /kaggle/working/Container-Shipping-Number2-1/test/labels... 37 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 37/37 1.1Kit/s 0.0s
val: New cache created: /kaggle/working/Container-Shipping-Number2-1/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 3/3 2.2it/s 1.4s
                   all         37         73      0.944          1      0.992      0.865
      container-number         37         37      0.949          1      0.992      0.871
              iso-type         36         36      0.939          1      0.992      0.858
Speed: 6.5ms preprocess, 11.6ms inference, 0.0ms loss, 1.2ms postprocess per image
Results saved to /kaggle/working/runs/c

In [9]:
from pathlib import Path
from ultralytics import YOLO
import matplotlib.pyplot as plt
from PIL import Image

BEST_MODEL_PATH = Path(
    "/kaggle/working/runs/container_code_type_yolo11n_v2/weights/best.pt"
)

TEST_IMAGES_DIR = Path(
    "/kaggle/working/Container-Shipping-Number2-1/test/images"
)

PREDICTION_PROJECT = Path("/kaggle/working/runs")
PREDICTION_NAME = "container_code_type_yolo11n_v2_predictions"

model_v2_best = YOLO(str(BEST_MODEL_PATH))

prediction_results = model_v2_best.predict(
    source=str(TEST_IMAGES_DIR),
    imgsz=640,
    conf=0.25,
    iou=0.7,
    device=0,
    save=True,
    project=str(PREDICTION_PROJECT),
    name=PREDICTION_NAME,
    exist_ok=False,
    verbose=True,
)

prediction_dir = PREDICTION_PROJECT / PREDICTION_NAME

print("Prédictions enregistrées dans :", prediction_dir)
print("Nombre d’images analysées :", len(prediction_results))

predicted_images = sorted(
    [
        path
        for path in prediction_dir.iterdir()
        if path.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"}
    ]
)[:9]

plt.figure(figsize=(18, 15))

for index, image_path in enumerate(predicted_images, start=1):
    image = Image.open(image_path)

    plt.subplot(3, 3, index)
    plt.imshow(image)
    plt.title(image_path.name)
    plt.axis("off")

plt.tight_layout()
plt.show()


image 1/37 /kaggle/working/Container-Shipping-Number2-1/test/images/Container-125_jpg.rf.039db97ad054b8107d01773904ef48f4.jpg: 640x640 1 container-number, 1 iso-type, 8.5ms
image 2/37 /kaggle/working/Container-Shipping-Number2-1/test/images/Container-129_jpg.rf.1f120222cddfb5402db1c75126b29147.jpg: 640x640 1 container-number, 1 iso-type, 9.4ms
image 3/37 /kaggle/working/Container-Shipping-Number2-1/test/images/Container-139_jpg.rf.74916dfd6a3d3d7016782dfe59ef2175.jpg: 640x640 1 container-number, 1 iso-type, 8.8ms
image 4/37 /kaggle/working/Container-Shipping-Number2-1/test/images/Container-151_jpg.rf.b94953973fe1d41427a73b7dfc4efeda.jpg: 640x640 1 container-number, 1 iso-type, 8.4ms
image 5/37 /kaggle/working/Container-Shipping-Number2-1/test/images/Container-159_jpg.rf.6aec9efeed5fe0d9da9a25ece6377ef2.jpg: 640x640 1 container-number, 1 iso-type, 10.7ms
image 6/37 /kaggle/working/Container-Shipping-Number2-1/test/images/Container-161_jpg.rf.a00b2d66a40d984566ff142a4302ea66.jpg: 640x64

<Figure size 1800x1500 with 9 Axes>

In [10]:
from pathlib import Path
import shutil
import json
import zipfile
import yaml

# ============================================================
# CHEMINS
# ============================================================

TRAIN_RUN_DIR = Path(
    "/kaggle/working/runs/container_code_type_yolo11n_v2"
)

TEST_RUN_DIR = Path(
    "/kaggle/working/runs/container_code_type_yolo11n_v2_test"
)

PREDICTION_RUN_DIR = Path(
    "/kaggle/working/runs/container_code_type_yolo11n_v2_predictions"
)

OUTPUT_ROOT = Path("/kaggle/working/output")

EXPORT_NAME = "marsatrack-yolo11-container-code-type-v2"
EXPORT_DIR = OUTPUT_ROOT / EXPORT_NAME
ZIP_PATH = OUTPUT_ROOT / f"{EXPORT_NAME}.zip"

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# Recréer proprement le dossier si la cellule est exécutée plusieurs fois
if EXPORT_DIR.exists():
    shutil.rmtree(EXPORT_DIR)

if ZIP_PATH.exists():
    ZIP_PATH.unlink()

EXPORT_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================
# SOUS-DOSSIERS
# ============================================================

MODELS_DIR = EXPORT_DIR / "models"
TRAINING_RESULTS_DIR = EXPORT_DIR / "training_results"
TEST_RESULTS_DIR = EXPORT_DIR / "test_results"
PREDICTIONS_DIR = EXPORT_DIR / "predictions"

MODELS_DIR.mkdir()
TRAINING_RESULTS_DIR.mkdir()
TEST_RESULTS_DIR.mkdir()
PREDICTIONS_DIR.mkdir()

# ============================================================
# MODÈLES
# ============================================================

BEST_MODEL_SOURCE = TRAIN_RUN_DIR / "weights" / "best.pt"
LAST_MODEL_SOURCE = TRAIN_RUN_DIR / "weights" / "last.pt"

BEST_MODEL_DESTINATION = (
    MODELS_DIR / "container_code_type_yolo11n_v2_best.pt"
)

LAST_MODEL_DESTINATION = (
    MODELS_DIR / "container_code_type_yolo11n_v2_last.pt"
)

if not BEST_MODEL_SOURCE.exists():
    raise FileNotFoundError(
        f"Modèle best.pt introuvable : {BEST_MODEL_SOURCE}"
    )

shutil.copy2(BEST_MODEL_SOURCE, BEST_MODEL_DESTINATION)

if LAST_MODEL_SOURCE.exists():
    shutil.copy2(LAST_MODEL_SOURCE, LAST_MODEL_DESTINATION)

# ============================================================
# COPIE DES RÉSULTATS D’ENTRAÎNEMENT
# ============================================================

training_files = [
    "args.yaml",
    "results.csv",
    "results.png",
    "confusion_matrix.png",
    "confusion_matrix_normalized.png",
    "BoxP_curve.png",
    "BoxR_curve.png",
    "BoxPR_curve.png",
    "BoxF1_curve.png",
    "labels.jpg",
    "labels_correlogram.jpg",
    "train_batch0.jpg",
    "train_batch1.jpg",
    "train_batch2.jpg",
    "val_batch0_labels.jpg",
    "val_batch0_pred.jpg",
    "val_batch1_labels.jpg",
    "val_batch1_pred.jpg",
    "val_batch2_labels.jpg",
    "val_batch2_pred.jpg",
]

for filename in training_files:
    source = TRAIN_RUN_DIR / filename

    if source.exists():
        shutil.copy2(source, TRAINING_RESULTS_DIR / filename)

# ============================================================
# COPIE DES RÉSULTATS DU TEST
# ============================================================

test_files = [
    "args.yaml",
    "confusion_matrix.png",
    "confusion_matrix_normalized.png",
    "BoxP_curve.png",
    "BoxR_curve.png",
    "BoxPR_curve.png",
    "BoxF1_curve.png",
    "val_batch0_labels.jpg",
    "val_batch0_pred.jpg",
    "val_batch1_labels.jpg",
    "val_batch1_pred.jpg",
    "val_batch2_labels.jpg",
    "val_batch2_pred.jpg",
]

for filename in test_files:
    source = TEST_RUN_DIR / filename

    if source.exists():
        shutil.copy2(source, TEST_RESULTS_DIR / filename)

# ============================================================
# COPIE DES PRÉDICTIONS VISUELLES
# ============================================================

prediction_extensions = {".jpg", ".jpeg", ".png", ".webp"}

prediction_count = 0

if PREDICTION_RUN_DIR.exists():
    for source in sorted(PREDICTION_RUN_DIR.iterdir()):
        if (
            source.is_file()
            and source.suffix.lower() in prediction_extensions
        ):
            shutil.copy2(source, PREDICTIONS_DIR / source.name)
            prediction_count += 1

# ============================================================
# MÉTRIQUES DU JEU DE TEST
# ============================================================

metrics = {
    "model": "YOLO11n",
    "version": "V2",
    "classes": [
        "container-number",
        "iso-type",
    ],
    "dataset": {
        "source_images": 745,
        "version_images": 1931,
        "train_images": 1779,
        "validation_images": 115,
        "test_images": 37,
        "container_number_boxes": 1973,
        "iso_type_boxes": 1749,
    },
    "test": {
        "images": 37,
        "instances": 73,
        "precision": float(test_metrics_v2.box.mp),
        "recall": float(test_metrics_v2.box.mr),
        "map50": float(test_metrics_v2.box.map50),
        "map50_95": float(test_metrics_v2.box.map),
        "per_class": {
            "container-number": {
                "precision": 0.949,
                "recall": 1.0,
                "map50": 0.992,
                "map50_95": 0.871,
            },
            "iso-type": {
                "precision": 0.939,
                "recall": 1.0,
                "map50": 0.992,
                "map50_95": 0.858,
            },
        },
    },
    "inference": {
        "image_size": 640,
        "average_preprocess_ms": 2.2,
        "average_inference_ms": 8.2,
        "average_postprocess_ms": 1.2,
    },
}

with open(
    EXPORT_DIR / "metrics.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        metrics,
        file,
        indent=2,
        ensure_ascii=False,
    )

# ============================================================
# INFORMATIONS DU MODÈLE
# ============================================================

model_summary = {
    "architecture": "YOLO11n",
    "task": "Object Detection",
    "classes_count": 2,
    "classes": [
        "container-number",
        "iso-type",
    ],
    "training_epochs": 70,
    "image_size": 640,
    "best_model": BEST_MODEL_DESTINATION.name,
    "last_model": (
        LAST_MODEL_DESTINATION.name
        if LAST_MODEL_DESTINATION.exists()
        else None
    ),
}

with open(
    EXPORT_DIR / "model_summary.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        model_summary,
        file,
        indent=2,
        ensure_ascii=False,
    )

# ============================================================
# README
# ============================================================

readme_content = """# MarsaTrack AI — YOLO11 Container Code & Type V2

## Objectif

Ce modèle détecte simultanément deux zones sur une image de conteneur maritime :

- `container-number` : matricule ISO 6346, par exemple `SESU2072393`;
- `iso-type` : code de taille et de type, par exemple `22G1`.

## Pipeline prévu

Image complète
→ détection YOLO11
→ recadrage des zones
→ OCR
→ validation du matricule ISO 6346
→ validation du code taille/type
→ correction manuelle possible
→ enregistrement dans MarsaTrack AI

## Dataset

- 745 images sources;
- 1 931 images après augmentation;
- 1 779 images d’entraînement;
- 115 images de validation;
- 37 images de test;
- 1 973 annotations `container-number`;
- 1 749 annotations `iso-type`.

## Résultats sur le jeu de test

- Précision globale : 94,4 %;
- Recall global : environ 100 %;
- mAP50 : 99,2 %;
- mAP50-95 : 86,5 %.

### container-number

- Précision : 94,9 %;
- Recall : 100 %;
- mAP50 : 99,2 %;
- mAP50-95 : 87,1 %.

### iso-type

- Précision : 93,9 %;
- Recall : 100 %;
- mAP50 : 99,2 %;
- mAP50-95 : 85,8 %.

## Modèle principal

`models/container_code_type_yolo11n_v2_best.pt`

Le modèle V1 consacré uniquement au matricule doit rester conservé séparément.
"""

with open(
    EXPORT_DIR / "README.md",
    "w",
    encoding="utf-8",
) as file:
    file.write(readme_content)

# ============================================================
# CRÉATION DU ZIP
# ============================================================

with zipfile.ZipFile(
    ZIP_PATH,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as zip_file:
    for file_path in sorted(EXPORT_DIR.rglob("*")):
        if file_path.is_file():
            archive_name = file_path.relative_to(OUTPUT_ROOT)
            zip_file.write(file_path, archive_name)

# ============================================================
# RAPPORT FINAL
# ============================================================

print("Sauvegarde V2 terminée")
print()
print("Dossier de sortie :", EXPORT_DIR)
print("Archive ZIP :", ZIP_PATH)
print(
    "Taille du meilleur modèle :",
    round(BEST_MODEL_DESTINATION.stat().st_size / 1024 / 1024, 2),
    "MB",
)
print(
    "Taille du ZIP :",
    round(ZIP_PATH.stat().st_size / 1024 / 1024, 2),
    "MB",
)
print("Prédictions sauvegardées :", prediction_count)

print("\nFichiers principaux :")

for file_path in sorted(EXPORT_DIR.rglob("*")):
    if file_path.is_file():
        print("-", file_path.relative_to(OUTPUT_ROOT))

Sauvegarde V2 terminée

Dossier de sortie : /kaggle/working/output/marsatrack-yolo11-container-code-type-v2
Archive ZIP : /kaggle/working/output/marsatrack-yolo11-container-code-type-v2.zip
Taille du meilleur modèle : 5.22 MB
Taille du ZIP : 20.79 MB
Prédictions sauvegardées : 37

Fichiers principaux :
- marsatrack-yolo11-container-code-type-v2/README.md
- marsatrack-yolo11-container-code-type-v2/metrics.json
- marsatrack-yolo11-container-code-type-v2/model_summary.json
- marsatrack-yolo11-container-code-type-v2/models/container_code_type_yolo11n_v2_best.pt
- marsatrack-yolo11-container-code-type-v2/models/container_code_type_yolo11n_v2_last.pt
- marsatrack-yolo11-container-code-type-v2/predictions/Container-125_jpg.rf.039db97ad054b8107d01773904ef48f4.jpg
- marsatrack-yolo11-container-code-type-v2/predictions/Container-129_jpg.rf.1f120222cddfb5402db1c75126b29147.jpg
- marsatrack-yolo11-container-code-type-v2/predictions/Container-139_jpg.rf.74916dfd6a3d3d7016782dfe59ef2175.jpg
- marsat